# 📦 TinyTransformer - Kaggle Cloud CPU Data Sharding Pipeline

**High-Throughput NVMe Binary Shard Generator on Kaggle CPU (Zero TPU Quota Consumed)**

### 💡 Why Run Sharding on Kaggle CPU First?
- **Saves Your TPU Quota:** Kaggle provides 30 hours/week of TPU quota. Running tokenization on CPU uses **0 TPU minutes**, preserving 100% of your TPU time for pure training.
- **Zero Network Bottlenecks:** Streams raw datasets from Hugging Face once and writes contiguous `uint16` memory-mapped binary shards directly to disk.
- **Turn Into Kaggle Dataset:** When this notebook finishes, save the output as a Kaggle Dataset (e.g. `tinytransformer-shards-600m`). In your TPU training notebook, attach this dataset for **instant 0-minute startup** at ~150,000 tokens/sec!

### 📐 Dataset Strategy & Volume (2.0 Billion Total Tokens):
- **Foundation Pretraining (1.85 Billion Tokens ~3.7 GB uint16 disk):**
  - 30% FineWeb-Edu Dedup (~555M tokens) [Core high-signal anchor]
  - 22% Cosmopedia v2 (~407M tokens) [Pedagogical synthetic reasoning]
  - 15% FineMath-3plus (~278M tokens) [Arithmetic & foundational algebra]
  - 12% Python-Edu (~222M tokens) [Structured syntax & algorithms]
  - 11% DCLM-100BT (~203M tokens) [Natural human web variance / anti-synthetic bias]
  - 8% Wikipedia (~148M tokens) [Factual entity grounding]
  - 2% Instruction Warmup (~37M tokens) [SmolTalk foundational alignment]
- **Instruction Alignment (150 Million Tokens ~300 MB uint16 disk):**
  - Multi-faceted SmolTalk mixture (Tool calling, reasoning CoT, coding, dialogue, Magpie, OpenHermes)
- **Optimal Epoch Count:** In an 8-hour TPU v5e-8 run (~3.8B–4.0B tokens processed), this provides **exactly 2.0 epochs** (~98% data efficiency per Muennighoff et al., NeurIPS 2023).

---
## ⚙️ Step 1: Clone Repository & Install Dependencies
Ensures you have the latest tokenization engine and tokenizer rules.

In [ ]:
import os

if not os.path.exists("/kaggle/working/TinyTransformer"):
    !git clone https://github.com/logicstillloading/TinyTransformer.git /kaggle/working/TinyTransformer

%cd /kaggle/working/TinyTransformer
!git pull origin main

# Install streaming & dataset dependencies
!pip install -q datasets tokenizers psutil

---
## 🌐 Step 2: Download & Tokenize Foundation Shards (Pretrain)
Streams web, mathematical, scientific, and coding corpora to create contiguous `uint16` NVMe binary shards.

In [ ]:
%cd /kaggle/working/TinyTransformer

# 🎛️ Foundation Pretraining Token Target: 1.85 Billion tokens (~3.7 GB disk)
# Yields ~2.0 epochs across our 3.8B - 4.0B TPU v5e-8 training run (~98% data efficiency per Muennighoff et al.)
FOUNDATION_TARGET_TOKENS = 1_850_000_000

# Runs parallel streaming tokenization for foundation streams
!python -u data/download_and_tokenize.py \
    --shard-dir /kaggle/working/TinyTransformer/data/shards \
    --phase foundation \
    --target-tokens {FOUNDATION_TARGET_TOKENS}

---
## 🤖 Step 3: Download & Tokenize Instruction Shards (Alignment)
Streams diverse agentic instruction, coding, reasoning, and tool-use conversations using XML special tokens.

In [ ]:
%cd /kaggle/working/TinyTransformer

# 🎛️ Instruction Alignment Token Target: 150 Million tokens (~300 MB disk)
# Covers multi-faceted agentic, reasoning, and tool-calling distillation
INSTRUCT_TARGET_TOKENS = 150_000_000

# Runs parallel streaming tokenization for multi-faceted instruction streams
!python -u data/download_and_tokenize.py \
    --shard-dir /kaggle/working/TinyTransformer/data/shards \
    --phase instruct \
    --target-tokens {INSTRUCT_TARGET_TOKENS}

---
## 📊 Step 4: Verify & Audit Shards on Disk
Verifies shard integrity, checks token totals per stream, and reports disk utilization.

In [ ]:
import os
import glob
from data.download_and_tokenize import get_shard_stats

stats = get_shard_stats("/kaggle/working/TinyTransformer/data/shards")
print("\n" + "=" * 70)
print("  [SHARD AUDIT REPORT]")
print("=" * 70)
print(f"* Total Tokens Processed: {stats['total_tokens']:,}")
print(f"* Total Disk Footprint:   {stats['total_mb']:.1f} MB ({stats['total_mb']/1024:.2f} GB)")
print("-" * 70)
for stream, s in stats["by_stream"].items():
    print(f"  • {stream:15s}: {s['tokens']:,} tokens | {s['shards']} shards ({s['mb']:.1f} MB)")
print("=" * 70 + "\n")

---
## 💾 Step 5: Export as Kaggle Dataset (1-Click Reuse for TPU)
Prepares your binary shards in `/kaggle/working/shards` so you can save this notebook version and create a reusable Kaggle Dataset.

In [ ]:
%cd /kaggle/working/TinyTransformer

# Automated Kaggle Dataset packaging with metadata generator and CLI push
!python scripts/create_kaggle_dataset.py \
    --shard-dir /kaggle/working/TinyTransformer/data/shards \
    --export-dir /kaggle/working/shards_export \
    --title "TinyTransformer Pretraining Shards (1.85B Foundation + 150M Instruct)" \
    --slug "tinytransformer-shards-2b"